# 06 - GRU Forecasting on Colab T4

Notebook nay duoc viet theo flow Colab-first:
- bat GPU T4 trong `Runtime > Change runtime type`
- clone repo vao `/content/hoankiem-air-quality-forecasting`
- import pipeline tu `src/`
- chay `smoke run` truoc khi train day du
- danh gia MAE / RMSE va ve actual vs predicted tren test


In [5]:
import os
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)

if not gpus:
    raise RuntimeError(
        'No GPU detected. In Colab, go to Runtime > Change runtime type > T4 GPU, then rerun from the top.'
    )

print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<not-set>'))
try:
    print('GPU name:', tf.config.experimental.get_device_details(gpus[0]).get('device_name', '<unknown>'))
except Exception as exc:
    print('Could not read GPU name:', exc)


TensorFlow version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CUDA_VISIBLE_DEVICES = <not-set>
GPU name: Tesla T4


In [8]:
%cd /content

!rm -rf hoankiem-air-quality-forecasting

!git clone https://github.com/HoangHumg1210/hoankiem-air-quality-forecasting.git

%cd hoankiem-air-quality-forecasting

!ls

/content
Cloning into 'hoankiem-air-quality-forecasting'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 95 (delta 41), reused 76 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 8.95 MiB | 15.51 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/hoankiem-air-quality-forecasting
data  notebooks  requirements.txt  src	tests  test_walkforward_data_utils.py


In [6]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/HoangHumg1210/hoankiem-air-quality-.git'
REPO_NAME = 'hoankiem-air-quality-forecasting'
PROJECT_ROOT = Path('/content') / REPO_NAME

os.chdir('/content')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_NAME], check=True)
else:
    print('Repo already exists at', PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_paths = [
    PROJECT_ROOT / 'src' / 'data_utils.py',
    PROJECT_ROOT / 'src' / 'gru_experiment.py',
    PROJECT_ROOT / 'data' / 'processed' / 'data2225_done.csv',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Missing required project files: ' + ', '.join(missing_paths))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('sys.path[0] =', sys.path[0])


Repo already exists at /content/hoankiem-air-quality-forecasting


FileNotFoundError: Missing required project files: /content/hoankiem-air-quality-forecasting/src/gru_experiment.py

In [ ]:
# Optional: run this cell only if a package import fails in the next cell.
# Colab already ships TensorFlow GPU, so do not reinstall tensorflow by default.
# %pip install -q -r requirements.txt


In [ ]:
# Optional: enable mixed precision after GPU detection is stable.
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy('mixed_float16')
# print('Mixed precision policy:', mixed_precision.global_policy())


In [9]:
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from src.data_utils import CFG, prepare_dataset
from src.gru_experiment import (
    GRUExperimentConfig,
    plot_forecast,
    plot_training_history,
    run_gru_experiment,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
print('TensorFlow sees GPU:', tf.config.list_physical_devices('GPU'))


ModuleNotFoundError: No module named 'src.gru_experiment'

In [ ]:
diag_artifacts = prepare_dataset(CFG)
print('Data path:', CFG.data_path)
print('Train seq:', diag_artifacts['X_train_seq'].shape, diag_artifacts['y_train_seq'].shape)
print('Val seq  :', diag_artifacts['X_val_seq'].shape, diag_artifacts['y_val_seq'].shape)
print('Test seq :', diag_artifacts['X_test_seq'].shape, diag_artifacts['y_test_seq'].shape)
print('n_features:', diag_artifacts['n_features'])


In [ ]:
data_cfg = CFG

smoke_cfg = GRUExperimentConfig(
    gru_units=(64,),
    dense_units=32,
    dropout=0.1,
    learning_rate=1e-3,
    batch_size=128,
    epochs=2,
    patience=2,
    step_to_plot=1,
)

full_cfg = GRUExperimentConfig(
    gru_units=(128, 64),
    dense_units=64,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=64,
    epochs=30,
    patience=8,
    step_to_plot=1,
)

print('Smoke config:', smoke_cfg)
print('Full config :', full_cfg)


In [ ]:
smoke_results = run_gru_experiment(data_cfg, smoke_cfg)
smoke_metrics = pd.concat(
    [smoke_results['val_results']['metrics_frame'], smoke_results['test_results']['metrics_frame']],
    ignore_index=True,
)
smoke_metrics


## Full training

Sau khi `smoke run` pass, chay cell duoi day de train day du tren Colab T4.


In [ ]:
results = run_gru_experiment(data_cfg, full_cfg)
artifacts = results['artifacts']
model = results['model']
history = results['history']
val_results = results['val_results']
test_results = results['test_results']


In [ ]:
print('Train seq:', artifacts['X_train_seq'].shape, artifacts['y_train_seq'].shape)
print('Val seq  :', artifacts['X_val_seq'].shape, artifacts['y_val_seq'].shape)
print('Test seq :', artifacts['X_test_seq'].shape, artifacts['y_test_seq'].shape)
model.summary()


In [7]:
metrics_df = pd.concat(
    [val_results['metrics_frame'], test_results['metrics_frame']],
    ignore_index=True,
)
metrics_df


NameError: name 'val_results' is not defined

In [ ]:
test_results['by_horizon'].head(15)


In [ ]:
plot_training_history(history)
plt.show()


In [ ]:
plot_forecast(
    test_results['forecast_frame'],
    title='GRU forecast on test',
)
plt.show()


In [ ]:
test_results['forecast_frame'].head(20)
